# 9. MCP (Model Context Protocol) Client

MCP lets tools, prompts, and resources live on a *separate server*, reached over
a standard protocol instead of a local Python import. This notebook connects to
this project's own MCP server (`mcp_server/`) and exercises all three primitives:
tools, prompts, and resources — plus both the hand-executed tool-calling round
(notebook 3's style) and the full `create_react_agent` loop (notebook 6's style),
this time with MCP-sourced tools.

**Prerequisites:** `scripts/start_mcp_server.sh` running (port `18383`), Ollama
running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

## Connect and list tools

In [ ]:
import os

from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_SERVER_URL = f"http://localhost:{os.getenv('MCP_SERVER_PORT', '18383')}/mcp"
client = MultiServerMCPClient({"ai_tutorial": {"transport": "streamable_http", "url": MCP_SERVER_URL}})

mcp_tools = await client.get_tools(server_name="ai_tutorial")
print([tool.name for tool in mcp_tools])

## Tools — one hand-executed round (same shape as notebook 3, but the tools came from MCP)

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

from models.chat_models.ollama_models import SupportedModel, get_chat_model

mcp_tools_by_name = {tool.name: tool for tool in mcp_tools}
llm = get_chat_model(SupportedModel.llama3_2)

messages = [HumanMessage(content="what is 12 times 7?")]
ai_message = llm.bind_tools(mcp_tools).invoke(messages)
messages.append(ai_message)

for tool_call in ai_message.tool_calls:
    result = await mcp_tools_by_name[tool_call["name"]].ainvoke(tool_call["args"])
    print(f"{tool_call['name']}({tool_call['args']}) = {result}")
    messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))

final_message = llm.invoke(messages)
print()
print("final answer:", final_message.content)

## Prompts — fetch the server's `explain_concept` prompt

In [ ]:
prompt_messages = await client.get_prompt("ai_tutorial", "explain_concept", arguments={"topic": "RAG"})
for message in prompt_messages:
    print(f"[{message.type}] {message.content}")

## Resources — read the dynamic models list and the static corpus

In [ ]:
models_resource = await client.get_resources("ai_tutorial", uris=["data://ai-tutorial/models"])
print("models:", models_resource[0].data)

corpus_resource = await client.get_resources("ai_tutorial", uris=["data://ai-tutorial/rag-corpus"])
print("corpus (first 150 chars):", corpus_resource[0].data[:150])

## Agent — MCP-sourced tools inside a full `create_react_agent` loop

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

AGENT_SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools. Once a tool call has "
    "returned a result that answers the user's question, respond directly "
    "with the final answer in plain text. Never call the same tool with the "
    "same arguments more than once."
)

mcp_agent = create_react_agent(llm, tools=mcp_tools, checkpointer=InMemorySaver(), prompt=AGENT_SYSTEM_PROMPT)
thread = {"configurable": {"thread_id": "notebook-demo"}}

agent_result = await mcp_agent.ainvoke({"messages": [("human", "what is 3+4?")]}, config=thread)
print(agent_result["messages"][-1].content)

In [ ]:
# Same thread_id — the agent should recall "7" from the previous turn.
agent_result_2 = await mcp_agent.ainvoke(
    {"messages": [("human", "what happens when I add 5 to it?")]}, config=thread
)
print(agent_result_2["messages"][-1].content)

## 🧪 Playground

**1. A string tool over MCP** — try `"reverse the word hello"` through the hand-executed round.

In [ ]:
# TODO: repeat the hand-executed tool-calling round with a string-tool question


**2. A fresh agent thread** — start a brand-new `thread_id` and ask a multi-part question in one turn.

In [ ]:
# TODO: new thread config + a question needing 2 tool calls in a single turn


**3. Fetch a different resource or prompt** — try `explain_concept` with a different `topic`, or think about what other prompt/resource you'd add to `mcp_server/` yourself.

In [ ]:
# TODO: fetch explain_concept with a different topic
